# Transitsuche bei Exoplaneten

Das Notebook lädt Kepler-Lichtkurven von Kepler-8, bereinigt instrumentelle Trends mit einem gleitenden Medianfilter und detektiert die Umlaufzeit mit dem Box Least Squares (BLS) Algorithmus. Anschließend wird ein physikalisches Transitmodell mit Randverdunkelung an die gefalteten Transits gefittet, um Planetenradius, Bahnneigung und Sterndichte zu bestimmen.

Wie bestimmt man aus den periodischen Helligkeitsschwankungen eines Sterns verlässlich die Umlaufzeit, den Radius und die mittlere Dichte eines Exoplaneten?

#### **Hintergrund**

Zieht ein Exoplanet von der Erde aus gesehen vor der Scheibe seines Zentralsterns vorüber, verdeckt er einen Bruchteil des Sternenlichts. Dieser periodische Helligkeitsabfall (Transit) ist eine der erfolgreichsten Methoden zur Entdeckung extrasolarer Planeten und bildet das Fundament der Weltraummissionen Kepler und TESS.

Die Form des Helligkeitseinbruchs enthält fundamentale physikalische Informationen:
- **Transittiefe $\delta$:** Gibt direkt das Flächenverhältnis $(R_p/R_\star)^2$ an.
- **Transitdauer $T$:** Bestimmt über Keplers drittes Gesetz die geometrische Bahngeschwindigkeit und Neigung.
- **Ingress und Egress:** Die Ein- und Austrittsphasen hängen vom Stoßparameter $b$ und der Randverdunkelung des Sterns ab.

Um solche periodischen Signale im Rauschen und vor dem Hintergrund stellarer Aktivität aufzuspüren, verwendet die Astronomie den **Box Least Squares (BLS)** Algorithmus. BLS modelliert Transits als periodische Kastenfunktionen und sucht über ein Frequenz- und Dauer-Gitter nach dem globalen Maximum der Signalstärke.

#### **Herleitung**

**1. Geometrische Transittiefe.** Verdeckt eine dunkle Planetenscheibe mit Radius $R_p$ eine gleichförmig leuchtende Sternscheibe mit Radius $R_\star$, sinkt der gemessene Fluss um
$$\delta = \frac{\Delta F}{F} = \left(\frac{R_p}{R_\star}\right)^2 = k^2.$$
Bei einer sonnenähnlichen Sternscheibe erzeugt ein jupitergroßer Planet einen Einbruch von $\approx 1\,\%$, ein erdgroßer Planet hingegen nur $\approx 84\,\text{ppm}$ ($8.4\cdot 10^{-5}$).

**2. Transitdauer und Stoßparameter.** Für eine kreisförmige Bahn mit großer Halbachse $a$ und Bahnneigung $i$ ist der projizierte Stoßparameter im Transitmittelpunkt definiert als
$$b = \frac{a}{R_\star} \cos i.$$
Ein Transit findet nur statt, wenn $b < 1 + k$ ist. Die Gesamtdauer zwischen erstem und viertem Kontakt ($T_{14}$) beträgt
$$T_{14} = \frac{P}{\pi} \arcsin\left( \frac{R_\star}{a} \frac{\sqrt{(1+k)^2 - b^2}}{\sin i} \right) \approx \frac{P}{\pi} \frac{R_\star}{a} \sqrt{(1+k)^2 - b^2}.$$

**3. Quadratische Randverdunkelung.** Reale Sterne strahlen am Rand kühler und dunkler ab als im Zentrum. Die Intensität $I(\mu)$ wird standardmäßig durch das quadratische Gesetz beschrieben:
$$I(\mu) = I(1) \left[ 1 - u_1(1-\mu) - u_2(1-\mu)^2 \right], \quad \mu = \cos\theta = \sqrt{1 - (r/R_\star)^2}.$$
Die Kipping-Parametrisierung $q_1 = (u_1 + u_2)^2$ und $q_2 = u_1 / (2(u_1 + u_2))$ garantiert physikalisch monotone und positive Profile im Einheitsquadrat $q_1, q_2 \in [0, 1]$.

**4. Mittlere Sterndichte aus Kepler III.** Aus Keplers drittem Gesetz $a^3 / P^2 = G (M_\star + M_p) / (4\pi^2) \approx G M_\star / (4\pi^2)$ folgt durch Division durch $R_\star^3$ direkt die mittlere Sterndichte:
$$\rho_\star = \frac{3 M_\star}{4\pi R_\star^3} = \frac{3\pi}{G P^2} \left(\frac{a}{R_\star}\right)^3.$$
Weil $a/R_\star$ und $P$ direkt aus der Lichtkurvenform gefittet werden, liefert die Transitanalyse eine reine photometrische Dichtemessung des Zentralsterns ohne Kenntnis seiner Masse oder Entfernung!

**5. Das BLS-Suchkriterium.** Für eine Zeitreihe mit Fluss $f_i$ und Gewichten $w_i = 1/\sigma_i^2$ testet BLS eine Kastenfunktion mit Tiefe $d$, Periode $P$ und Phasenintervall $[\phi_1, \phi_2]$. Das minimale $\chi^2$ entspricht der Maximierung der Teststatistik
$$S = \frac{s^2}{r(1-r)}, \quad s = \sum_{i \in \text{in}} w_i (f_i - \bar{f}), \quad r = \frac{\sum_{i \in \text{in}} w_i}{\sum_i w_i}.$$

#### **Daten**

Die Photometriedaten stammen aus der primären Kepler-Mission (NASA/MAST). Wir untersuchen den bestätigten Gasriesen **Kepler-8 b** (Entdeckung 2010), der einen F-Stern in einer 3.52-Tage-Bahn umkreist.

In [ ]:
%pip install -q lightkurve astroquery

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import lightkurve as lk
from astropy import units as u
from astroquery.ipac.nexsci.nasa_exoplanet_archive import NasaExoplanetArchive
from scipy.optimize import curve_fit
from astropy.timeseries import BoxLeastSquares

# Lokaler Cache
ASTRODATA = Path.cwd() / "astrodata"
for sub in ("astropy", "lightkurve"):
    (ASTRODATA / sub).mkdir(parents=True, exist_ok=True)
os.environ["XDG_CACHE_HOME"] = str(ASTRODATA)

SELECTED_PLANET = "Kepler-8 b"

cat = NasaExoplanetArchive.query_criteria(
    table="pscomppars", where=f"pl_name='{SELECTED_PLANET}'",
    select="pl_name, hostname, pl_orbper, pl_rade, pl_bmasse, st_rad, st_mass, st_dens, pl_ratror, pl_ratdor, pl_imppar, pl_trandur"
).to_pandas().iloc[0]

print(f"✅ Exoplanet:      {cat['pl_name']} (Zentralstern: {cat['hostname']})")
print(f"Katalog-Periode:   {cat['pl_orbper']:.5f} d")
print(f"Katalog-Radius:    {cat['pl_rade']:.2f} R_Erde  (Sternradius: {cat['st_rad']:.2f} R_Sonne)")
print(f"Sterndichte:       {cat['st_dens']:.3f} g/cm³")

#### **Reduktion**

##### *1. Rohdaten und Detrending*

Wie trennt man instrumentelle Niederfrequenz-Drifts des Teleskops von den scharfen Einbrüchen der Planetentransits?

In [ ]:
# Suche und Download eines Kepler-Quartals (Target Pixel File & Lichtkurve)
search = lk.search_targetpixelfile(SELECTED_PLANET, author="Kepler")
tpf = search[0].download()

lc_raw = tpf.to_lightcurve(aperture_mask=tpf.pipeline_mask)
lc = lc_raw.remove_nans().remove_outliers(sigma=5).normalize()

# Gleitender Savitzky-Golay / Median-Filter für das Detrending
# Fenster muss deutlich länger als die Transitdauer (~3 h), aber kürzer als die Umlaufzeit (3.5 d) sein
lc_flat = lc.flatten(window_length=101)

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
lc.plot(ax=axes[0], c="grey", alpha=0.8, label="Normalisierter Rohfluss")
axes[0].set_title(f"{SELECTED_PLANET} - Kepler-Lichtkurve vor und nach Detrending")
axes[0].set_ylabel("Fluss")
axes[0].legend()

lc_flat.plot(ax=axes[1], c="tab:green", alpha=0.8, label="Detrendeter Fluss")
axes[1].set_xlabel("Zeit [BJD - 2454833]")
axes[1].set_ylabel("Fluss")
axes[1].legend()
fig.tight_layout()
plt.show()

#### **Eigene Messung**

##### *2. Das BLS-Periodogramm*

Bei welcher Periode und Epoche zeigt die detrendete Lichtkurve das stärkste periodische Box-Signal?

In [ ]:
durations = np.linspace(0.05, 0.25, 15)  # Test-Transitdauern in Tagen (~1.2 bis 6 Stunden)

# Grobe BLS-Suche über Perioden von 1 bis 15 Tagen
pg_coarse = lc_flat.to_periodogram(
    method="bls",
    minimum_period=1.0,
    maximum_period=15.0,
    frequency_factor=5.0,
    duration=durations
)

best_p_coarse = float(pg_coarse.period_at_max_power.value)

# Feine Suche um den stärksten Peak
pg_fine = lc_flat.to_periodogram(
    method="bls",
    minimum_period=best_p_coarse * 0.98,
    maximum_period=best_p_coarse * 1.02,
    frequency_factor=20.0,
    duration=durations
)

period = pg_fine.period_at_max_power
t0 = pg_fine.transit_time_at_max_power
duration = pg_fine.duration_at_max_power
depth = pg_fine.depth_at_max_power

fig, ax = plt.subplots(figsize=(10, 4))
pg_fine.plot(ax=ax, c="black", lw=1)
ax.axvline(period.value, color="tab:red", ls="--", label=f"Beste Periode P = {period.value:.4f} d")
ax.set_xlabel("Periode [Tage]")
ax.set_ylabel("BLS Power (SNR)")
ax.set_title(f"BLS-Periodogramm für {SELECTED_PLANET}")
ax.legend()
plt.show()

print(f"✅ BLS-Detektion:")
print(f"  Periode P:     {period.value:.5f} d  (Katalog: {cat['pl_orbper']:.5f} d)")
print(f"  Transitmitte:  T0 = {t0.value:.4f} BJD")
print(f"  Dauer:         T14 = {duration.value * 24:.2f} Stunden")
print(f"  Tiefe:         delta = {depth.value * 1e6:.0f} ppm  ({depth.value * 100:.3f} %)")

##### *3. Phasengefaltete Lichtkurve*

Wie verhält sich das Transitsignal, wenn alle individuellen Transits auf die gefundene Periode gefaltet und gemittelt werden?

In [ ]:
folded = lc_flat.fold(period=period, epoch_time=t0)

fig, ax = plt.subplots(figsize=(10, 4.5))
folded.scatter(ax=ax, c="lightgrey", s=1.5, label="Messpunkte")
folded.bin(time_bin_size=duration / 8).plot(ax=ax, c="tab:red", lw=2, label="Gebinnte Lichtkurve")
ax.set_xlim(-3 * duration.value, 3 * duration.value)
ax.set_xlabel("Phase [Tage von Transitmitte]")
ax.set_ylabel("Normalisierter Fluss")
ax.set_title(f"{SELECTED_PLANET} - Phasengefaltet auf P = {period.value:.4f} d")
ax.legend()
plt.show()

#### **Modell und Fit**

##### *4. Physikalischer Transit-Fit mit Randverdunkelung*

Welcher Radius $R_p/R_\star$, Stoßparameter $b$ und welche Sterndichte $\rho_\star$ passen am besten zur gemessenen Kurvenform?

In [ ]:
G_CGS = 6.674e-8

def transit_flux(z, p, u1, u2, n_r=400):
    r = (np.arange(n_r) + 0.5) / n_r
    mu = np.sqrt(1 - r ** 2)
    I = 1 - u1 * (1 - mu) - u2 * (1 - mu) ** 2
    z = np.atleast_1d(z)[:, None]
    cos_th = np.clip((r ** 2 + z ** 2 - p ** 2) / (2 * r * np.maximum(z, 1e-12)), -1, 1)
    theta = np.where(r + z <= p, np.pi, np.where(np.abs(r - z) >= p, 0.0, np.arccos(cos_th)))
    return 1 - (I * theta * r).sum(axis=1) / (np.pi * (I * r).sum())

def transit_model(t, dt0, p, a_rs, b, q1, q2, period_d, exptime_d, n_sub=5):
    u1, u2 = 2 * np.sqrt(q1) * q2, np.sqrt(q1) * (1 - 2 * q2)
    inc = np.arccos(np.clip(b / a_rs, 0, 1))
    offsets = (np.arange(n_sub) - (n_sub - 1) / 2) / n_sub * exptime_d
    phase = 2 * np.pi * (np.asarray(t)[:, None] + offsets - dt0).ravel() / period_d
    z = a_rs * np.sqrt(np.sin(phase) ** 2 + (np.cos(inc) * np.cos(phase)) ** 2)
    z = np.where(np.cos(phase) > 0, z, 10.0)
    return transit_flux(z, p, u1, u2).reshape(len(t), n_sub).mean(axis=1)

def fold_phase(t, P, t0):
    return (t - t0 + 0.5 * P) % P - 0.5 * P

# Stacken mehrerer Lichtkurven für hohes Signal-Rausch-Verhältnis
lc_search = lk.search_lightcurve(SELECTED_PLANET, author="Kepler")
stitched = lc_search[:4].download_all().stitch()

unmask = lambda q: np.asarray(getattr(q, "unmasked", q).value, dtype=float)
t_all, f_all, e_all = unmask(stitched.time), unmask(stitched.flux), unmask(stitched.flux_err)
valid = np.isfinite(t_all) & np.isfinite(f_all) & np.isfinite(e_all)
t_all, f_all, e_all = t_all[valid], f_all[valid], e_all[valid]

P_d, t0_d = period.value, t0.value
T14 = duration.value
exptime_d = 1765.5 / 86400  # Kepler Long Cadence (~29.4 min)

# Detrending der gestitchten Zeitreihe mit Transitmaskierung
out_transit = np.abs(fold_phase(t_all, P_d, t0_d)) > 0.75 * T14
s = pd.Series(f_all[out_transit], index=pd.to_datetime(t_all[out_transit], unit="D"))
trend = s.rolling("0.8D", center=True, min_periods=5).median().to_numpy()
f_flat = f_all / np.interp(t_all, t_all[out_transit], trend)

ph = fold_phase(t_all, P_d, t0_d)
win = np.abs(ph) < 2.5 * T14
t_fit, f_fit, e_fit = ph[win], f_flat[win], e_all[win]

# Phasenbinning (2-Minuten-Intervalle)
bin_d = 2 / 1440
edges = np.arange(-2.5 * T14, 2.5 * T14 + bin_d, bin_d)
idx = np.digitize(t_fit, edges) - 1
nbins = len(edges) - 1
n_in = np.bincount(idx, minlength=nbins)[:nbins]
s1 = np.bincount(idx, weights=f_fit, minlength=nbins)[:nbins]
s2 = np.bincount(idx, weights=f_fit ** 2, minlength=nbins)[:nbins]
ok = n_in >= 3
tb = (0.5 * (edges[1:] + edges[:-1]))[ok]
fb = s1[ok] / n_in[ok]
eb = np.sqrt(np.maximum(s2[ok] / n_in[ok] - fb ** 2, 1e-12) / (n_in[ok] - 1))

model_fn = lambda t, dt0, p, a_rs, b, q1, q2: transit_model(t, dt0, p, a_rs, b, q1, q2, P_d, exptime_d, n_sub=5)
bounds = ([-0.02, 0.01, 1.5, 0.0, 0.0, 0.0], [0.02, 0.3, 30.0, 1.0, 1.0, 1.0])
p0 = [0.0, np.sqrt(depth.value), 7.0, 0.6, 0.35, 0.3]

popt, pcov = curve_fit(model_fn, tb, fb, p0=p0, sigma=eb, bounds=bounds, maxfev=10000)
perr = np.sqrt(np.diag(pcov))
dt0_fit, p_fit, a_fit, b_fit, q1_fit, q2_fit = popt

# Mittlere Sterndichte
rho_fit = 3 * np.pi * a_fit ** 3 / (G_CGS * (P_d * 86400) ** 2)
rho_err = rho_fit * 3 * perr[2] / a_fit

tt = np.linspace(-2.5 * T14, 2.5 * T14, 500)
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
axes[0].scatter(t_fit * 24, f_fit, s=1, c="lightgrey", label="Kadenzen")
axes[0].errorbar(tb * 24, fb, eb, fmt="o", c="black", ms=3, label="Gebinnt (2 min)")
axes[0].plot(tt * 24, model_fn(tt, *popt), c="tab:red", lw=2, label=f"Transitmodell: Rp/R* = {p_fit:.4f}, b = {b_fit:.2f}")
axes[0].set_ylabel("Normalisierter Fluss")
axes[0].set_title(f"{SELECTED_PLANET} - Physikalischer Transit-Fit mit Randverdunkelung")
axes[0].legend(fontsize=8)

axes[1].errorbar(tb * 24, (fb - model_fn(tb, *popt)) * 1e6, eb * 1e6, fmt=".", c="grey", ms=2)
axes[1].axhline(0, color="tab:red", lw=1)
axes[1].set_xlabel("Stunden von Transitmitte")
axes[1].set_ylabel("Residuum [ppm]")
fig.tight_layout()
plt.show()

#### **Eigene Messung gegen den Katalog**

##### *5. Vergleich mit den archivierten Werten*

Wie stimmen die gemessene Periode, der Planetenradius und die Sterndichte mit den archivierten Werten des NASA Exoplanet Archive überein?

In [ ]:
R_p_meas = p_fit * cat['st_rad'] * 109.08  # R_Erde
R_p_cat = cat['pl_rade']
P_cat = cat['pl_orbper']

print(f"{'Parameter':<25} {'Eigene Messung':<20} {'Katalogwert':<20} {'Abweichung':<15}")
print("-" * 75)
print(f"{'Periode P [Tage]':<25} {P_d:<20.5f} {P_cat:<20.5f} {abs(P_d - P_cat) / P_cat * 100:.3f} %")
print(f"{'Radiusverhältnis Rp/R*':<25} {p_fit:<20.4f} {cat['pl_ratror']:<20.4f} {abs(p_fit - cat['pl_ratror']) / cat['pl_ratror'] * 100:.2f} %")
print(f"{'Skalierte Halbachse a/R*':<25} {a_fit:<20.2f} {cat['pl_ratdor']:<20.2f} {abs(a_fit - cat['pl_ratdor']) / cat['pl_ratdor'] * 100:.2f} %")
print(f"{'Planetenradius [R_Erde]':<25} {R_p_meas:<20.2f} {R_p_cat:<20.2f} {abs(R_p_meas - R_p_cat) / R_p_cat * 100:.2f} %")
print(f"{'Sterndichte rho_* [g/cm³]':<25} {rho_fit:<20.3f} {cat['st_dens']:<20.3f} {abs(rho_fit - cat['st_dens']) / cat['st_dens'] * 100:.2f} %")

#### **Variationen**

Jeweils zuerst die Vorhersage notieren, dann den Code anpassen und testen:

1. **Detrending-Fensterlänge:** Verkleinere `window_length` in Schritt 1 von 101 auf 21 (ca. 10 Stunden). Was passiert mit der Transittiefe $\delta$? *(Vorhersage: Das Filter beginnt die Flanken des Transits als niederfrequenten Trend zu interpretieren und verflacht das Transitsignal).*
2. **Harmonische Perioden:** Starte die BLS-Suche in einem Intervall, das die halbe oder doppelte Periode ($P/2$ bzw. $2P$) erzwingt. Welches Signal-zu-Rausch-Verhältnis erreicht die Kastenfunktion im Vergleich zur Grundperiode?
3. **Randverdunkelung ausschalten:** Setze $q_1 = 0, q_2 = 0$ (flache Sternscheibe). Wie verändert sich das Residuum während Ingress und Egress?

#### **Nächste Schritte**

So würde eine professionelle Entdeckungsanalyse fortgesetzt:
- **MCMC-Sampling:** Vollständige bayessche Parameterinferenz (z. B. mit `emcee` oder `batman`), um die starke Entartung zwischen Stoßparameter $b$, Halbachse $a/R_\star$ und Randverdunkelungskoeffizienten in Wahrscheinlichkeitsdichten (Posterior-Verteilungen) aufzulösen.
- **Transit Timing Variations (TTV):** Vermessung individueller Transitzeiten $t_0^{(k)}$ jedes Umlaufs, um gravitative Störungen durch weitere, nicht-transidierende Planeten im System nachzuweisen.
- **Transit Least Squares (TLS):** Für kleine erdgroße Planeten liefert TLS eine höhere Nachweisempfindlichkeit als BLS, da es das typische U-Profil eines Transits statt einer Rechteckbox nutzt.

#### **Referenzen**

- Kovács, G., Zucker, S. & Mazeh, T. (2002): *A transit detection algorithm for survey data*, A&A, 391, 369–377 (BLS-Grundlagen).
- Mandel, K. & Agol, E. (2002): *Analytic Light Curves for Planetary Transit Searches*, ApJ, 580, L171.
- Kipping, D. M. (2013): *Efficient, uninformative sampling of limb darkening coefficients for two-parameter laws*, MNRAS, 435, 2152–2160.
- Jenkins, J. M. et al. (2010): *Overview of the Kepler Science Processing Pipeline*, ApJ, 713, L87.